In [1]:
import pandas as pd


In [3]:
attack_df = pd.read_csv("CTU-13.csv")
benign_df = pd.read_csv("Benign_Traffic_CIC2017.csv")

print("ATTACK SHAPE:", attack_df.shape)
print("BENIGN SHAPE:", benign_df.shape)


ATTACK SHAPE: (5933, 84)
BENIGN SHAPE: (2273097, 84)


In [4]:
attack_cols = list(attack_df.columns)
benign_cols = list(benign_df.columns)

print("Columns only in ATTACK:", set(attack_cols) - set(benign_cols))
print("Columns only in BENIGN:", set(benign_cols) - set(attack_cols))


Columns only in ATTACK: {'Fwd Pkts/s', 'Bwd Pkts/s', 'Bwd IAT Tot'}
Columns only in BENIGN: {'Bwd IAT Total', 'Fwd Packets/s', 'Bwd Packets/s'}


In [5]:
benign_df = benign_df.rename(columns={
    "Bwd IAT Total": "Bwd IAT Tot",
    "Fwd Packets/s": "Fwd Pkts/s",
    "Bwd Packets/s": "Bwd Pkts/s"
})


In [6]:
benign_df = benign_df[attack_df.columns]


In [7]:
assert list(attack_df.columns) == list(benign_df.columns)
print("✅ Column names and order match")


✅ Column names and order match


In [8]:
print("Attack labels:", attack_df["Label"].unique())
print("Benign labels:", benign_df["Label"].unique()[:10])


Attack labels: [1]
Benign labels: ['BENIGN']


In [9]:
benign_df['Label'] = 0

In [10]:
print("Attack labels:", attack_df["Label"].unique())
print("Benign labels:", benign_df["Label"].unique()[:10])


Attack labels: [1]
Benign labels: [0]


In [11]:
assert benign_df["Label"].dtype == attack_df["Label"].dtype
print("✅ Label column normalized")


✅ Label column normalized


In [12]:
canonical_dtypes = attack_df.dtypes.to_dict()


In [13]:
for col, target_dtype in canonical_dtypes.items():
    if col == "Label":
        continue

    if benign_df[col].dtype != target_dtype:
        if target_dtype == "int64":
            benign_df[col] = pd.to_numeric(benign_df[col], errors="raise").astype("int64")
        elif target_dtype == "float64":
            benign_df[col] = pd.to_numeric(benign_df[col], errors="raise").astype("float64")
        else:
            benign_df[col] = benign_df[col].astype("object")


In [14]:
for col in attack_df.columns:
    assert attack_df[col].dtype == benign_df[col].dtype, \
        f"Dtype mismatch in column: {col}"

print("✅ Full schema match: names, order, dtypes")


✅ Full schema match: names, order, dtypes


In [15]:
n_attack = len(attack_df)

benign_sample = benign_df.sample(
    n=n_attack,
    random_state=42
)


In [16]:
combined_df = pd.concat(
    [attack_df, benign_sample],
    ignore_index=True
)

combined_df = combined_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)


In [17]:
print("FINAL SHAPE:", combined_df.shape)
print(combined_df["Label"].value_counts())
print(combined_df.dtypes.value_counts())


FINAL SHAPE: (11866, 84)
Label
1    5933
0    5933
Name: count, dtype: int64
float64    45
int64      35
object      4
Name: count, dtype: int64


In [18]:
combined_df.to_csv("balanced_CTU13_attack_benign.csv", index=False)
print("✅ File saved: balanced_CTU13_attack_benign.csv")


✅ File saved: balanced_CTU13_attack_benign.csv
